Create summary tables for results after broad and manual filtering of MBR data

In [1]:
import glob
import os
import pandas as pd
from collections import defaultdict

In [2]:
meta = pd.read_csv('../mbr_studies_metadata.csv')
meta = meta.set_index('study_id')
meta.head()

,study_name,paper_title,author_list,study_species,study_species_common,num_individuals,num_gps_points,timespan,start_date,end_date,continent,paper_url,main_csv_file,chatgpt_responses,broad_taxonomic_group
study_id,,,,,,,,,,,,,,,
aae3de22-4736-45c0-bfcb-af0a4d57ba0f,Byholm_aae3de22,"Study ""GPS tracking of honey buzzards in Finland""","Byholm, Patrik and Mirski, Paweł and Vansteela...",Pernis apivorus,European honey-buzzard,28,72848,7.2 years,2011-07-29,2018-09-25,Africa,https://doi.org/10.5441/001/1.335,GPS tracking of honey buzzards in Finland.csv(...,"[{""row_index"": 1, ""publication_url"": ""https://...",avian
6f734b0f-7a0a-4f9d-af97-1d33fce053db,Wagner_6f734b0f,"Abundance, diet and foraging of Galápagos Barn...","Wagner, Hermann and Cruz, Sebastian and Jiméne...",Tyto furcata,American Barn Owl,8,25274,3.7 years,2016-03-01,2019-11-19,South America,https://doi.org/10.5441/001/1.699,Galapagos barn owl project.csv(3.26 MB).csv,"[{""row_index"": 2, ""publication_url"": ""https://...",avian
166c0893-a7bc-4b40-93b3-9337a7e01c6d,SorianoVarela_166c0893,"Study ""Foraging movement patterns of lactating...","Soriano-Varela, Paulina and Ibarra-Macías, Ana...",Leptonycteris nivalis,Mexican long-nosed bat,21,1282,1 year,2021-05-24,2022-06-08,South America,https://doi.org/10.5441/001/1.696,Foraging movement patterns of lactating Mexica...,"[{""row_index"": 3, ""publication_url"": ""https://...",terrestrial mammals
fb867768-11eb-4b55-9460-6ada78cc34c4,Hebblewhite_fb867768,"Study ""Hebblewhite Alberta-BC Wolves""","Hebblewhite, Mark",Canis lupus,Gray Wolf,68,174454,11.4 years,2000-01-25,2011-06-22,North America,https://doi.org/10.5441/001/1.662,Hebblewhite Alberta-BC Wolves.csv(24.93 MB).csv,"[{""row_index"": 4, ""publication_url"": ""https://...",terrestrial mammals
376eb55c-2284-4b48-be3b-c0ef1e41cb57,Noboa_376eb55c,Home range reduction of Tapirus terrestris rai...,"Noboa, Mateo and Castellanos, Francisco X. and...",Tapirus terrestris,Brazilian tapir,1,606,7 months,2023-07-16,2024-02-28,South America,https://doi.org/10.5441/001/1.677,Home range reduction of Tapirus terrestris rai...,NaN,terrestrial mammals


In [3]:
studies37 = """9eaf99f2-9a3b-4b48-a680-aa74542e162a
df53968d-4bcc-4dd7-9bb6-8358378836ec
71fc09c6-6127-4884-991f-d4e144e282a7
75d6171c-d981-4bdf-bf23-bf2af17a7e47
eda74cd3-d35b-4fb5-9981-2d72b3e641d5
4654cef6-e6c3-44ce-9884-d55c3c662464
dce434f5-0cae-45be-8847-e45b61833bf0
e2fe5582-a707-4954-8329-ba107736fed8
6360747f-37f5-40f1-9731-f4926387a21e
338388ea-6483-472c-9cdc-c2997e645df0
8fed4a87-14d3-4f9d-b630-e1d406b15c68
e7808391-1019-45cb-8701-e3ded043e0d8
86f58f48-90be-4241-b895-6af3e7952dde
5a280544-06db-4c85-9713-842547972edd
fd28aac2-b22f-4368-8638-59906e769ded
3cef8e9d-c82a-43d2-8bea-fd9d3527c66b
f565e4ce-a459-4bc7-8d3c-11a386901c97
b624221e-33ef-4844-bd66-f007fab78e6f
28708a9c-5582-490e-9481-c094bdce74c2
2facc53d-e454-4bd4-82d1-f17040ec42a3
0b812dab-840b-4940-af5d-98cf2c5f9f63
e3df6f82-0508-4b4b-a168-6a579176441e
8318572a-a9d8-4d9e-b685-7db8a73bdf86
a41015db-4632-46dc-abed-277b16daf97b
f577908f-52ae-46a1-986a-1862336cee91
91804022-7b95-4192-a1bb-5288e1141225
831030a3-c255-46a8-8de8-e8a786d32dc6
a398b445-fb68-47a5-9dd9-9ca259ea2c20
ae7b0b8c-d025-40fc-9c19-8c42c14228a0
cec3bcf7-08fe-43e7-ac0c-af8f7d684a04
5c3ce9e4-c498-4d72-bf7e-40b91c60a024
97abc24d-afff-49f0-8a89-56bc8c6c5308
c5d8e38f-ab54-498e-8372-efdc8af6c0f9
10ea2b4f-84e0-46c2-b731-28ec0598d98f
7de63cc4-8212-4584-8dc2-b970079ac36f
c73c7698-2c6b-44d4-8879-fe10d7536935
6f734b0f-7a0a-4f9d-af97-1d33fce053db""".split('\n')

studies72 = studies37 + """aae3de22-4736-45c0-bfcb-af0a4d57ba0f
a8224f36-fc72-4b91-bb13-684df2247792
3e431f5b-7248-460a-a800-c5788d3c9fa3
520a64ae-2047-445d-af5f-7b961caeef79
6d60b346-369c-46be-b976-d793b61fa076
5f8c5199-c2b0-4e48-9521-8c5d5f6b115c
0a3ec70a-967c-432a-99b6-794c4f6b63f0
79710d0e-d1d6-45fd-90f9-1c6d9be0c46e
a1d2d3ab-adcd-4101-8775-e0559445b612
5de9f888-c77f-444d-a397-ac05128a1e86
26c82518-d17c-470c-8c9a-557d1b7fbe61
5672a811-db2a-472f-bc42-35dd41ee5ea9
fd78ade4-cedd-459c-b151-fac2ca2f65a1
e82ecc51-ad30-4e52-b956-47b51a3ee19a
a19440b7-58a0-488a-a345-97f8d7afef87
319926e6-3b42-4864-a761-192ed681dbe1
386d7ca8-7a05-49ca-a081-740688506e98
346009bb-5f5f-4363-890b-2e5119f78ed0
12d8637e-80b8-4850-994f-71f6d6734007
aa260e37-8ff6-4366-9918-91346785b8f9
9d8e2a5a-3c17-4898-8231-511beb8c044c
08b6b850-df1f-4740-b05e-acb1a654934b
bc8dc065-69df-4002-aa34-7c3eca71109b
22d42221-334b-4eb5-88b7-628c6492bacb
e4070707-8471-45f2-8e15-7ca0cd8a9ada
10bb3544-713b-473b-a7e1-2cf5553bbaa9
86f824b7-1973-4ba8-bc6a-fa8956e830a1
29312801-d75d-418a-b4ce-8c5d0ad5b045
1e31df42-edfa-4225-b923-d8b0de83ab20
88c61e4c-e277-46ae-9762-2e2d9efbf451
e68f2847-c651-47f3-84e5-c429ec8e27d9
d52006ff-9c80-4c9e-8170-6569f8018f39
86008da5-779d-4d96-ac83-66652b552402
3695ca1b-4a59-473a-a0f7-3aa5773ec022
4cf33db8-277d-4402-b6ff-06e99de6d0e0""".split('\n')

In [4]:
output_dir = '../mbr_studies_2year'
files = sorted(glob.glob(os.path.join(output_dir, '*_2year.csv')))

# --- NEW: read text files ---------------------------------------------

def _read_list(path):
    if not os.path.exists(path):
        return set()
    with open(path, "r", encoding="utf-8") as f:
        return {ln.strip() for ln in f if ln.strip()}

ok_files_path = os.path.join(output_dir, 'ok_files.txt')
not_ok_files_path = os.path.join(output_dir, 'not_ok_files.txt')
bad_individuals_path = os.path.join(output_dir, 'bad_individuals.txt')

ok_files = _read_list(ok_files_path)
not_ok_files = _read_list(not_ok_files_path)

# bad_individuals.txt: "<csv_basename>,<individual-id>"
bad_individuals_by_file = defaultdict(set)
if os.path.exists(bad_individuals_path):
    with open(bad_individuals_path, "r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = ln.split(",", 1)
            if len(parts) != 2:
                continue
            fname, indiv = parts
            bad_individuals_by_file[fname].add(str(indiv))

# ----------------------------------------------------------------------

if not files:
    print('No processed files found in', output_dir)
else:
    total_rows = 0
    unique_individuals = set()
    file_stats = []
    group_stats = {}

    for file_path in files:
        filename = os.path.basename(file_path)

        # --- NEW: skip studies explicitly marked as "not ok" -----------
        if filename in not_ok_files:
            # This study is meant to be removed entirely
            continue

        try:
            df = pd.read_csv(file_path, usecols=['individual-local-identifier'])
        except ValueError:
            df = pd.read_csv(file_path)
        except Exception as exc:
            print(f'Skipping {file_path} due to error: {exc}')
            continue

        if 'individual-local-identifier' not in df.columns:
            print(f'Skipping {file_path}: missing individual-local-identifier column')
            continue

        # --- NEW: skip studies with > 600k GPS points ------------------
        if len(df) > 600_000:
            print(f"Skipping {file_path}: >600k GPS points ({len(df):,})")
            continue
        # ----------------------------------------------------------------

        # --- NEW: drop individuals marked as bad for this file ----------
        bad_here = bad_individuals_by_file.get(filename, set())
        if bad_here:
            df['individual-local-identifier'] = df['individual-local-identifier'].astype(str)
            df = df[~df['individual-local-identifier'].isin(bad_here)].copy()

        # If everything in this file was bad, it contributes zero
        num_rows = len(df)
        if num_rows == 0:
            continue

        num_individuals = df['individual-local-identifier'].nunique()

        total_rows += num_rows
        study_id = filename.split('_', 1)[0]
        identifiers = df['individual-local-identifier'].dropna().astype(str).unique()

        unique_individuals.update((study_id, identifier) for identifier in identifiers)
        file_stats.append((filename, num_rows, num_individuals))

        if study_id in meta.index:
            base_group = meta.loc[study_id, 'broad_taxonomic_group']
        else:
            base_group = 'Unknown'

        # Default group name
        group_name = base_group

        # # Refine Avian into short/medium vs long distance
        # if base_group == 'avian':
        #     # Short-to-medium distance: study_id in studies37
        #     if study_id in studies37:
        #         group_name = 'Avian: Short-to-medium distance'
        #     # Long distance: study_id in studies72 but NOT in studies37
        #     elif study_id in studies72 and study_id not in studies37:
        #         group_name = 'Avian: Long distance'
        #     # else: leave as plain 'Avian' if it’s not in either list

        stats = group_stats.setdefault(group_name, {'studies': set(), 'individuals': set(), 'rows': 0})
        stats['studies'].add(study_id)
        stats['rows'] += num_rows
        for identifier in identifiers:
            stats['individuals'].add((study_id, identifier))

    print('Per-file totals (rows, individuals):')
    for name, rows, individuals in file_stats:
        print(f'  {name}: rows={rows:,}, individuals={individuals:,}')

    print('\nOVERALL TOTALS:')
    print(f'Total rows: {total_rows:,}')
    print(f'Total unique individuals: {len(unique_individuals):,}')

    print('\nTotals by broad taxonomic group:')
    for group_name, stats in sorted(group_stats.items()):
        print(
            f"  {group_name}: studies={len(stats['studies'])}, "
            f"individuals={len(stats['individuals'])}, rows={stats['rows']:,}"
        )


/tmp/ipykernel_777554/1804384496.py:52: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, usecols=['individual-local-identifier'])


Skipping ../mbr_studies_2year/86f824b7-1973-4ba8-bc6a-fa8956e830a1_vanToor_86f824b7_2year.csv: >600k GPS points (804,024)


/tmp/ipykernel_777554/1804384496.py:52: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, usecols=['individual-local-identifier'])


Skipping ../mbr_studies_2year/aa260e37-8ff6-4366-9918-91346785b8f9_Schwemmer_aa260e37_2year.csv: >600k GPS points (649,983)
Skipping ../mbr_studies_2year/f577908f-52ae-46a1-986a-1862336cee91_Jennings_f577908f_2year.csv: >600k GPS points (2,951,415)
Per-file totals (rows, individuals):
  08b6b850-df1f-4740-b05e-acb1a654934b_Mskens_08b6b850_2year.csv: rows=36,718, individuals=1
  0b812dab-840b-4940-af5d-98cf2c5f9f63_Beck_0b812dab_2year.csv: rows=83,100, individuals=20
  10bb3544-713b-473b-a7e1-2cf5553bbaa9_Byholm_10bb3544_2year.csv: rows=122,180, individuals=3
  1e31df42-edfa-4225-b923-d8b0de83ab20_Batbayar_1e31df42_2year.csv: rows=64,731, individuals=3
  22d42221-334b-4eb5-88b7-628c6492bacb_Klzsch_22d42221_2year.csv: rows=15,767, individuals=1
  26c82518-d17c-470c-8c9a-557d1b7fbe61_Batbayar_26c82518_2year.csv: rows=36,216, individuals=3
  28708a9c-5582-490e-9481-c094bdce74c2_Carter_28708a9c_2year.csv: rows=21,397, individuals=3
  2facc53d-e454-4bd4-82d1-f17040ec42a3_Scasta_2facc53d_2yea

In [9]:
# -----------------------------
# Build LaTeX summary table
# -----------------------------

# Collect rows for table
latex_rows = []
total_studies = 0
total_indivs = 0
total_rows_count = 0

for group_name, stats in sorted(group_stats.items()):
    n_studies = len(stats['studies'])
    n_indivs = len(stats['individuals'])
    n_rows = stats['rows']

    total_studies += n_studies
    total_indivs += n_indivs
    total_rows_count += n_rows

    latex_rows.append(
        f"{group_name} & {n_studies} & {n_indivs} & {n_rows:,} \\\\"
    )

# Total row
latex_rows.append(
    f"\\midrule\nTotal & {total_studies} & {total_indivs} & {total_rows_count:,} \\\\"
)

# Full LaTeX table
latex_table = (
    "\\begin{table}[h!]\n"
    "    \\centering\n"
    "    \\begin{tabular}{l|r|r|r}\n"
    "         High-level group & n studies & n indivs & n gps \\\\\n"
    "         \\midrule\n"
    + "\n".join(latex_rows) +
    "\n"
    "    \\end{tabular}\n"
    "    \\caption{{Final filtering results.}}\n"
    "    \\label{tab:studies-final}\n"
    "\\end{table}"
)

print("\nLaTeX-formatted summary table:\n")
print(latex_table)



LaTeX-formatted summary table:

\begin{table}[h!]
    \centering
    \begin{tabular}{l|r|r|r}
         High-level group & n studies & n indivs & n gps \\
         \midrule
avian & 24 & 221 & 2,035,556 \\
reptiles & 2 & 38 & 455,398 \\
terrestrial mammals & 11 & 126 & 704,689 \\
\midrule
Total & 37 & 385 & 3,195,643 \\
    \end{tabular}
    \caption{{Final filtering results.}}
    \label{tab:studies-final}
\end{table}


In [6]:
# Create species-level summary table with LaTeX output using actual taxon data
import glob
output_dir = '../mbr_studies_2year'
files = sorted(glob.glob(os.path.join(output_dir, '*_2year.csv')))

species_summary = []

for file_path in files:
    try:
        # Get study ID from filename
        filename = os.path.basename(file_path)
        study_id = filename.split('_', 1)[0]

        # Skip huge studies
        try:
            n_rows_full = sum(1 for _ in open(file_path)) - 1
        except Exception:
            n_rows_full = None

        if n_rows_full is not None and n_rows_full > 600_000:
            continue


        if study_id not in meta.index:
            continue

        # Get broad taxonomic group from metadata
        study_row = meta.loc[study_id]
        broad_group = study_row['broad_taxonomic_group']

        # Read the processed file to get actual species data
        df = pd.read_csv(file_path, usecols=['individual-local-identifier', 'individual-taxon-canonical-name'])
        
        # Group by species and count individuals and GPS points
        species_counts = df.groupby('individual-taxon-canonical-name').agg({
            'individual-local-identifier': ['nunique', 'count']
        }).reset_index()
        
        # Flatten column names
        species_counts.columns = ['species', 'n_individuals', 'n_gps']
        
        # Add metadata
        for _, row in species_counts.iterrows():
            species_summary.append({
                'study_id': study_id,
                'species': row['species'],
                'broad_group': broad_group,
                'n_studies': 1,
                'n_individuals': row['n_individuals'],
                'n_gps': row['n_gps']
            })

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        continue

# Convert to DataFrame and aggregate by species across all studies
species_df = pd.DataFrame(species_summary)

# Aggregate by species (sum studies, individuals, and GPS points)
species_agg = species_df.groupby(['species', 'broad_group']).agg({
    'n_studies': 'sum',
    'n_individuals': 'sum',
    'n_gps': 'sum'
}).reset_index()

# Create high-level group mapping for better formatting
group_mapping = {
    'avian': 'Avian',
    'terrestrial mammals': 'Terrestrial mammals',
    'reptiles': 'Reptiles',
    'humans/domestic': 'Domestic \\& Agriculture'
}

species_agg['high_level_group'] = species_agg['broad_group'].map(group_mapping)

# Sort by high level group alphabetically, then by species name alphabetically within group
species_agg = species_agg.sort_values(['high_level_group', 'species'])

# Generate LaTeX table
print("\\begin{table}[h!]")
print("    \\centering")
print("    \\begin{tabular}{l|l|r|r|r}")
print("         High-level group & Species & n studies & n indivs & n gps \\\\")
print("         \\midrule")

current_group = None
for _, row in species_agg.iterrows():
    group = row['high_level_group']
    species = row['species'].replace('_', '\\_')  # Escape underscores for LaTeX
    studies = row['n_studies']
    individuals = row['n_individuals']
    gps = f"{row['n_gps']:,}"

    # Only show group name for first occurrence
    if group != current_group:
        group_display = group
        current_group = group
    else:
        group_display = ""

    print(f"         {group_display} & {species} & {studies} & {individuals} & {gps}\\\\")

# Add totals row
total_studies = species_agg['n_studies'].sum()
total_individuals = species_agg['n_individuals'].sum()
total_gps = f"{species_agg['n_gps'].sum():,}"

print("         \\midrule")
print(f"         Total & & {total_studies} & {total_individuals} & {total_gps}")
print("    \\end{tabular}")
print("    \\label{tab:species_summary}")
print("\\end{table}")

# Also show summary by high-level group for reference
print("\n\nSummary by high-level group:")
group_summary = species_agg.groupby('high_level_group').agg({
    'n_studies': 'sum',
    'n_individuals': 'sum',
    'n_gps': 'sum'
}).reset_index()

for _, row in group_summary.iterrows():
    group = row['high_level_group']
    studies = row['n_studies']
    individuals = row['n_individuals']
    gps = f"{row['n_gps']:,}"
    print(f"{group}: {studies} studies, {individuals} individuals, {gps} GPS points")

/tmp/ipykernel_777554/700953555.py:32: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, usecols=['individual-local-identifier', 'individual-taxon-canonical-name'])


\begin{table}[h!]
    \centering
    \begin{tabular}{l|l|r|r|r}
         High-level group & Species & n studies & n indivs & n gps \\
         \midrule
         Avian & Anser albifrons & 4 & 34 & 122,493\\
          & Anser anser & 2 & 26 & 161,018\\
          & Anthropoides virgo & 5 & 15 & 79,967\\
          & Aquila nipalensis & 1 & 16 & 160,008\\
          & Bartramia longicauda & 1 & 2 & 3,155\\
          & Cathartes aura & 1 & 30 & 279,958\\
          & Coragyps atratus & 1 & 7 & 217,283\\
          & Eudocimus albus & 1 & 19 & 82,587\\
          & Grus grus & 2 & 6 & 79,278\\
          & Grus nigricollis & 1 & 3 & 107,796\\
          & Grus vipio & 2 & 7 & 74,364\\
          & Hydroprogne caspia & 1 & 4 & 145,127\\
          & Larus canus & 1 & 2 & 3,899\\
          & Larus smithsonianus & 1 & 11 & 235,303\\
          & Milvus milvus & 1 & 4 & 81,121\\
          & Morus bassanus & 1 & 3 & 126,178\\
          & Pernis apivorus & 1 & 2 & 7,793\\
          & Puffinus puffinus & 1 &

In [7]:
import glob
import os
import pandas as pd
from collections import defaultdict

output_dir = '../mbr_studies_2year'
files = sorted(glob.glob(os.path.join(output_dir, '*_2year.csv')))

# -------------------------------------------------------------------
# Helpers to read text files (OK / NOT OK / bad individuals)
# -------------------------------------------------------------------

def _read_list(path):
    if not os.path.exists(path):
        return set()
    with open(path, "r", encoding="utf-8") as f:
        return {ln.strip() for ln in f if ln.strip()}

ok_files_path = os.path.join(output_dir, 'ok_files.txt')
not_ok_files_path = os.path.join(output_dir, 'not_ok_files.txt')
bad_individuals_path = os.path.join(output_dir, 'bad_individuals.txt')

ok_files = _read_list(ok_files_path)
not_ok_files = _read_list(not_ok_files_path)

# bad_individuals.txt: "<csv_basename>,<individual-id>"
bad_individuals_by_file = defaultdict(set)
if os.path.exists(bad_individuals_path):
    with open(bad_individuals_path, "r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = ln.split(",", 1)
            if len(parts) != 2:
                continue
            fname, indiv = parts
            bad_individuals_by_file[fname].add(str(indiv))

# -------------------------------------------------------------------
# Build global species -> common-name mapping from meta
#   using study_species and study_species_common (semicolon-aligned)
# -------------------------------------------------------------------

species_to_common = {}

if 'study_species' in meta.columns and 'study_species_common' in meta.columns:
    for study_id, row in meta.iterrows():
        sci_str = row['study_species']
        com_str = row['study_species_common']

        if pd.isna(sci_str) or pd.isna(com_str):
            continue

        sci_list = [s.strip() for s in str(sci_str).split(';')]
        com_list = [c.strip() for c in str(com_str).split(';')]

        n = min(len(sci_list), len(com_list))
        for sci, com in zip(sci_list[:n], com_list[:n]):
            if sci and com and sci not in species_to_common:
                species_to_common[sci] = com

species_summary = []

for file_path in files:
    try:
        filename = os.path.basename(file_path)
        study_id = filename.split('_', 1)[0]

        # Skip huge studies
        try:
            n_rows_full = sum(1 for _ in open(file_path)) - 1
        except Exception:
            n_rows_full = None

        if n_rows_full is not None and n_rows_full > 600_000:
            continue

        # Skip studies not in meta
        if study_id not in meta.index:
            continue

        # Skip studies explicitly marked as NOT OK
        if filename in not_ok_files:
            continue

        study_row = meta.loc[study_id]
        broad_group = study_row['broad_taxonomic_group']

        # Refine Avian into short/medium vs long distance
        # assumes studies37 = short/medium, studies72 = long-distance sets
        if broad_group == 'avian':
            if study_id in studies37:
                broad_group = 'avian_short'
            elif study_id in studies72 and study_id not in studies37:
                broad_group = 'avian_long'
            # else leave as plain 'avian' if it’s not in either list

        # Read processed file: individuals + species
        try:
            df = pd.read_csv(
                file_path,
                usecols=['individual-local-identifier', 'individual-taxon-canonical-name']
            )
        except ValueError:
            df = pd.read_csv(file_path)

        if 'individual-local-identifier' not in df.columns or \
           'individual-taxon-canonical-name' not in df.columns:
            continue

        # Drop individuals marked as bad for this file
        bad_here = bad_individuals_by_file.get(filename, set())
        if bad_here:
            df['individual-local-identifier'] = df['individual-local-identifier'].astype(str)
            df = df[~df['individual-local-identifier'].isin(bad_here)].copy()

        if df.empty:
            continue

        df['individual-local-identifier'] = df['individual-local-identifier'].astype(str)

        # Group by species and count individuals and GPS points
        species_counts = df.groupby('individual-taxon-canonical-name').agg({
            'individual-local-identifier': ['nunique', 'count']
        }).reset_index()

        # Flatten column names
        species_counts.columns = ['species', 'n_individuals', 'n_gps']

        # Add metadata (per species in this study)
        for _, row in species_counts.iterrows():
            species_summary.append({
                'study_id': study_id,
                'species': row['species'],
                'broad_group': broad_group,
                'n_studies': 1,
                'n_individuals': row['n_individuals'],
                'n_gps': row['n_gps']
            })

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        continue

# -------------------------------------------------------------------
# Convert to DataFrame and aggregate by species across all (kept) studies
# -------------------------------------------------------------------
species_df = pd.DataFrame(species_summary)

if species_df.empty:
    print("No data available for species summary (after filtering).")
else:
    # Consolidate: one row per (species, broad_group)
    species_agg = species_df.groupby(['species', 'broad_group'], dropna=False).agg({
        'n_studies': 'sum',
        'n_individuals': 'sum',
        'n_gps': 'sum'
    }).reset_index()

    # Add common name from global mapping
    species_agg['common_name'] = species_agg['species'].map(
        lambda s: species_to_common.get(s, "")
    )

    # Map refined broad groups to display names
    group_mapping = {
        'avian': 'Avian (unspecified distance)',
        'avian_short': 'Avian: Short-to-medium distance',
        'avian_long': 'Avian: Long distance',
        'terrestrial mammals': 'Terrestrial mammals',
        'reptiles': 'Reptiles',
        'humans/domestic': 'Domestic \\& Agriculture'
    }

    species_agg['high_level_group'] = species_agg['broad_group'].map(group_mapping).fillna('Unknown')

    # Sort by high-level group, then species name
    species_agg = species_agg.sort_values(['high_level_group', 'species'])

    # -----------------------------------------------------------------
    # Generate LaTeX table
    # -----------------------------------------------------------------
    from textwrap import wrap

    def latex_wrap(text, width=25):
        """Wrap text into multiple lines for LaTeX using \\makecell."""
        if text is None:
            return ""
        text = str(text)
        lines = wrap(text, width=width)
        safe = [ln.replace('_', '\\_') for ln in lines]
        return "\\makecell{" + " \\\\ ".join(safe) + "}"

    print("\\begin{table}[h!]")
    print("    \\centering")
    print("    \\begin{tabular}{l|l|l|r|r}")
    print("         High-level group & Species (scientific) & Common name & n indivs & n gps \\\\")
    print("         \\midrule")

    current_group = None
    for _, row in species_agg.iterrows():
        group = latex_wrap(row['high_level_group'])
        species = latex_wrap(row['species'])
        common = latex_wrap(row['common_name'])
        individuals = row['n_individuals']
        gps = f"{row['n_gps']:,}"

        if group != current_group:
            if current_group is not None:
                print("         \\midrule")
            group_display = group
            current_group = group
        else:
            group_display = ""

        print(f"         {group_display} & {species} & {common} & {individuals} & {gps}\\\\")
        
    # Totals row
    total_individuals = species_agg['n_individuals'].sum()
    total_gps = f"{species_agg['n_gps'].sum():,}"

    print("         \\midrule")
    print(f"         Total & & & {total_individuals} & {total_gps}")
    print("    \\end{tabular}")
    print("    \\label{tab:species_summary}")
    print("\\end{table}")


/tmp/ipykernel_777554/840501523.py:101: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


\begin{table}[h!]
    \centering
    \begin{tabular}{l|l|l|r|r}
         High-level group & Species (scientific) & Common name & n indivs & n gps \\
         \midrule
         \makecell{Avian: Long distance} & \makecell{Anser albifrons} & \makecell{White-fronted goose} & 2 & 52,485\\
          & \makecell{Anthropoides virgo} & \makecell{Anthropoides virgo} & 15 & 79,967\\
          & \makecell{Aquila nipalensis} & \makecell{steppe eagle} & 15 & 160,008\\
          & \makecell{Bartramia longicauda} & \makecell{upland sandpiper} & 2 & 3,155\\
          & \makecell{Cathartes aura} & \makecell{turkey vulture} & 30 & 279,958\\
          & \makecell{Coragyps atratus} & \makecell{black vulture} & 7 & 217,283\\
          & \makecell{Grus grus} & \makecell{Common Crane} & 5 & 60,464\\
          & \makecell{Grus nigricollis} & \makecell{black-necked crane} & 3 & 107,796\\
          & \makecell{Grus vipio} & \makecell{Grus vipio} & 7 & 74,364\\
          & \makecell{Hydroprogne caspia} & \makecel

----

The following cell prints quick summaries of all the plots for all the studies, included in the appendix.

In [8]:
# --------- Emit LaTeX -------------------------------------------------

for high_group in sorted(groups.keys()):
    print(f"\\subsection{{{high_group}}}")
    print()

    for info in sorted(groups[high_group], key=lambda x: x["study_name"]):

        filename = info["filename"]
        study_id = info["study_id"]
        study_name = info["study_name"]

        # ---- PAPER TITLE from meta ----
        paper_title = meta.loc[study_id].get("paper_title", study_id)
        if pd.isna(paper_title) or str(paper_title).strip() == "":
            paper_title = study_id
        latex_title = str(paper_title).replace("_", "\\_")

        # ---- SPECIES LIST for this study (scientific + common) ----
        sci_str = meta.loc[study_id].get("study_species", "")
        com_str = meta.loc[study_id].get("study_species_common", "")

        species_lines = []
        if isinstance(sci_str, str) and isinstance(com_str, str):
            sci_list = [s.strip() for s in sci_str.split(";") if s.strip()]
            com_list = [c.strip() for c in com_str.split(";") if c.strip()]
            m = min(len(sci_list), len(com_list))
            for sci, com in zip(sci_list[:m], com_list[:m]):
                species_lines.append(f"{sci} ({com})")

        species_text = ", ".join(species_lines) if species_lines else "Species unknown"

        # ---- LaTeX subsubsection ----
        print(f"\\subsubsection{{{latex_title}}}")
        print()

        # ---- Figure ----
        png_path = f"{png_dir}/{study_name}_summary_plots.png"
        print("\\begin{figure}[h!]")
        print("    \\centering")
        print(f"    \\includegraphics[width=0.95\\textwidth]{{{png_path}}}")
        # print(f"    \\caption{{Summary plots for {latex_title}.}}")
        print("\\end{figure}")
        print()

        # ---- Text summary ----
        n_indiv = info["n_indiv"]
        n_gps = f"{info['n_gps']:,}"
        keep_status = info["keep_status"]

        # Individuals removed
        if info["removed_indiv"]:
            removed_str = ", ".join(info["removed_indiv"])
            indiv_text = f"Some individuals were removed: {removed_str}."
        else:
            indiv_text = "All individuals were kept."

        # Split overrides
        if info["split_overrides"]:
            changes = ", ".join(f"{indiv} on {date}" for indiv, date in info["split_overrides"])
            split_text = (
                f"Train/test split was changed for the following individuals: {changes}."
            )
        else:
            split_text = (
                "Train/test split was left at the default midpoint for all individuals."
            )

        summary = (
            f"This study includes the following species: {species_text}. "
            f"It has {n_indiv} individuals and {n_gps} GPS points. "
            f"The study was {keep_status}. "
            f"{indiv_text} {split_text}"
        )

        print("\\noindent " + summary)
        print(" \clearpage")
        print()
        print("% -----------------------------------------------------")
        print()


NameError: name 'groups' is not defined